# 🎥 Video 07: Molecular Fingerprints Deep Dive
**RDKit Masterclass — From Zero to Drug Discovery AI**
* **Duration:** ~25 - 30 Minutes
* **Curriculum Level:** LEVEL 3 — Molecular Fingerprints (Topics 13, 14, 15, 16, 17)
* **Target Audience:** AI/ML Engineers, Cheminformatics Researchers, Data Scientists

---

### ⏱️ Video Agenda & Timestamps
* **00:00 - 05:00** | Descriptors vs Fingerprints: Molecular Properties vs Structural Bit Patterns
* **05:00 - 10:00** | Circular Fingerprints: The Morgan / ECFP Algorithm Explained
* **10:00 - 14:30** | ECFP4 vs ECFP6: Radius, Bit Size, Hashing & Bit Collisions
* **14:30 - 18:30** | Modern RDKit API: `rdFingerprintGenerator.GetMorganGenerator`
* **18:30 - 22:30** | Structural Keys: MACCS Keys (166 predefined bits)
* **22:30 - 26:30** | Topological Fingerprints: RDKit Daylight-like & Atom Pairs
* **26:30 - 30:00** | Converting to NumPy Arrays for Scikit-Learn & PyTorch

---

### 🎯 Learning Objectives
1. Understand why fixed-length bit vectors are necessary for chemical machine learning.
2. Master circular Extended Connectivity Fingerprints (ECFP) and choose between radius 2 (ECFP4) and radius 3 (ECFP6).
3. Use the modern `rdFingerprintGenerator` architecture introduced in recent RDKit releases.
4. Calculate structural key fingerprints (MACCS 166 keys) and compare with Morgan fingerprints.
5. Convert RDKit bit vectors into dense NumPy arrays for machine learning pipelines.

In [1]:
# Step 0: Imports and Setup
import rdkit
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator, MACCSkeys, RDKFingerprint
from rdkit import DataStructs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"RDKit Version: {rdkit.__version__}")

RDKit Version: 2026.03.6


## 1. Descriptors vs Molecular Fingerprints
* **Physicochemical Descriptors (e.g. MW, LogP, TPSA):** Aggregate global properties. Two completely different structures (e.g. a long aliphatic chain vs an aromatic ring) can have identical MW and LogP.
* **Molecular Fingerprints:** Encode the presence or absence of specific **substructural fragment patterns** as binary bits ($0$ or $1$) in a high-dimensional vector space (e.g. 1024 or 2048 bits).

Fingerprints enable:
1. Fast chemical similarity search.
2. Molecular clustering and diversity analysis.
3. Feature vectors for Machine Learning classifiers and regressors.

In [2]:
# Let's inspect Aspirin and Paracetamol
aspirin = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")
paracetamol = Chem.MolFromSmiles("CC(=O)Nc1ccc(O)cc1")

print(f"Aspirin heavy atoms:     {aspirin.GetNumAtoms()}")
print(f"Paracetamol heavy atoms: {paracetamol.GetNumAtoms()}")

Aspirin heavy atoms:     13
Paracetamol heavy atoms: 11


## 2. Morgan / ECFP Fingerprints Explained
The **Extended Connectivity Fingerprint (ECFP)** algorithm:
1. Assigns initial integer identifiers to each atom based on atomic number, charge, valence, etc.
2. Iteratively updates each atom's identifier by hashing the identifiers of its immediate neighbors in concentric spheres of radius $r$.
3. At iteration $r=1$: captures immediate atom pairs (radius 1, diameter 2 -> ECFP2).
4. At iteration $r=2$: captures 2-bond neighborhoods (radius 2, diameter 4 -> **ECFP4**).
5. Hashes the resulting integer IDs into a fixed-length bit vector (e.g. 1024 or 2048 bits) using modulo arithmetic.

In [3]:
# Modern RDKit API: rdFingerprintGenerator
# ECFP4 (Radius = 2, Length = 2048)
morgan_gen_r2 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fp_aspirin_r2 = morgan_gen_r2.GetFingerprint(aspirin)

# ECFP6 (Radius = 3, Length = 2048)
morgan_gen_r3 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)
fp_aspirin_r3 = morgan_gen_r3.GetFingerprint(aspirin)

print(f"ECFP4 (radius 2) active 'on' bits: {fp_aspirin_r2.GetNumOnBits()} / 2048")
print(f"ECFP6 (radius 3) active 'on' bits: {fp_aspirin_r3.GetNumOnBits()} / 2048")

ECFP4 (radius 2) active 'on' bits: 24 / 2048
ECFP6 (radius 3) active 'on' bits: 31 / 2048


## 3. Count Vectors vs Bit Vectors
A standard **Bit Vector** only records whether a fragment is present ($0$ or $1$).
A **Count Vector** records how many times that exact fragment appears (e.g. 3 methyl groups = count 3), preserving stoichiometry.

In [4]:
# Generate Count Fingerprint
fp_count = morgan_gen_r2.GetCountFingerprint(aspirin)
non_zero_elements = fp_count.GetNonzeroElements()
print(f"Number of unique fragment environments: {len(non_zero_elements)}")
print("Sample (bit_id: count):", dict(list(non_zero_elements.items())[:5]))

Number of unique fragment environments: 24
Sample (bit_id: count): {389: 1, 456: 1, 650: 2, 695: 1, 807: 3}


## 4. MACCS Keys (166 Predefined Structural Patterns)
**MACCS (Molecular ACCess System) Keys** are 166 predefined SMARTS patterns (e.g., bit 125 = aromatic ring, bit 84 = halogen, bit 155 = OH group).
Unlike Morgan fingerprints, there are **no hash collisions** in MACCS keys, and every bit has a fixed, documented chemical meaning!

In [5]:
maccs_aspirin = MACCSkeys.GenMACCSKeys(aspirin)
maccs_paracetamol = MACCSkeys.GenMACCSKeys(paracetamol)

print(f"MACCS Keys Total Bits: {maccs_aspirin.GetNumBits()}")
print(f"Aspirin 'on' bits:     {maccs_aspirin.GetNumOnBits()}")
print(f"Paracetamol 'on' bits: {maccs_paracetamol.GetNumOnBits()}")

MACCS Keys Total Bits: 167
Aspirin 'on' bits:     21
Paracetamol 'on' bits: 23


## 5. Topological & Atom-Pair Fingerprints
* **RDKit Topological (Daylight-like):** Hashes linear paths of atoms and bonds up to a default length of 7.
* **Atom Pair Fingerprints:** Encodes pairs of atoms and the shortest topological path distance between them (e.g., `C_sp3` separated by 4 bonds from `O_sp2`).

In [6]:
# RDKit Topological Fingerprint
rdk_gen = rdFingerprintGenerator.GetRDKitFPGenerator(minPath=1, maxPath=7, fpSize=2048)
fp_rdk = rdk_gen.GetFingerprint(aspirin)

# Atom Pair Fingerprint
atom_pair_gen = rdFingerprintGenerator.GetAtomPairGenerator(minDistance=1, maxDistance=30, fpSize=2048)
fp_atompair = atom_pair_gen.GetFingerprint(aspirin)

print(f"RDKit Topological 'on' bits: {fp_rdk.GetNumOnBits()}")
print(f"Atom Pair 'on' bits:         {fp_atompair.GetNumOnBits()}")

RDKit Topological 'on' bits: 354
Atom Pair 'on' bits:         68


## 6. Converting RDKit Fingerprints to NumPy Arrays
For integration with **Scikit-Learn**, **XGBoost**, or **PyTorch**, you need dense or sparse NumPy arrays.

In [7]:
def fp_to_numpy(rdkit_fp):
    """Converts an ExplicitBitVect into a 1D numpy array of float32."""
    arr = np.zeros((rdkit_fp.GetNumBits(),), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(rdkit_fp, arr)
    return arr

arr_aspirin = fp_to_numpy(fp_aspirin_r2)
print(f"NumPy array shape: {arr_aspirin.shape}")
print(f"NumPy array dtype: {arr_aspirin.dtype}")
print(f"First 25 bits:     {arr_aspirin[:25].astype(int)}")

NumPy array shape: (2048,)
NumPy array dtype: float32
First 25 bits:     [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


## 7. 🎯 Video Hands-On Challenge & Solution
### Problem:
Take 3 structurally distinct drugs:
* **Gefitinib** (Kinase inhibitor, multi-ring aromatic)
* **Penicillin V** (Beta-lactam antibiotic)
* **Ibuprofen** (Small NSAID)
1. Generate ECFP4 (Morgan r=2), MACCS keys, and RDKit Topological fingerprints for each.
2. Calculate the bit density (`on_bits / total_bits`) for each fingerprint type.
3. Compare their bit overlap.

In [8]:
challenge_drugs = {
    "Gefitinib": "COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN4CCOCC4",
    "Penicillin V": "CC1(C(N2C(S1)C(C2=O)NC(=O)COc3ccccc3)C(=O)O)C",
    "Ibuprofen": "CC(C)Cc1ccc(cc1)C(C)C(=O)O"
}

summary = []
for name, smi in challenge_drugs.items():
    m = Chem.MolFromSmiles(smi)
    fp_ecfp4 = morgan_gen_r2.GetFingerprint(m)
    fp_maccs = MACCSkeys.GenMACCSKeys(m)
    fp_rdk = rdk_gen.GetFingerprint(m)
    
    summary.append({
        "Drug": name,
        "HeavyAtoms": m.GetNumAtoms(),
        "ECFP4_OnBits": fp_ecfp4.GetNumOnBits(),
        "ECFP4_Density": f"{fp_ecfp4.GetNumOnBits() / 2048:.2%}",
        "MACCS_OnBits": fp_maccs.GetNumOnBits(),
        "MACCS_Density": f"{fp_maccs.GetNumOnBits() / 166:.2%}",
        "RDKit_OnBits": fp_rdk.GetNumOnBits(),
        "RDKit_Density": f"{fp_rdk.GetNumOnBits() / 2048:.2%}"
    })

df_fp = pd.DataFrame(summary)
print("=== 🔬 FINGERPRINT COMPARISON TABLE ===")
display(df_fp)

=== 🔬 FINGERPRINT COMPARISON TABLE ===


,Drug,HeavyAtoms,ECFP4_OnBits,ECFP4_Density,MACCS_OnBits,MACCS_Density,RDKit_OnBits,RDKit_Density
0,Gefitinib,31,60,2.93%,60,36.14%,901,43.99%
1,Penicillin V,24,46,2.25%,59,35.54%,1072,52.34%
2,Ibuprofen,15,25,1.22%,15,9.04%,228,11.13%


## 8. Summary & Key Takeaways
* **Morgan / ECFP4** is the gold standard fingerprint for molecular property prediction, QSAR, and similarity searching.
* **`rdFingerprintGenerator.GetMorganGenerator()`** is the modern thread-safe API replacing legacy functions.
* **MACCS Keys** provide fixed, interpretable structural features (166 bits) without bit collisions.
* `DataStructs.ConvertToNumpyArray()` bridges the gap between RDKit and Python Machine Learning ecosystems.

**Next Up in Video 08:** **Molecular Similarity & Similarity Searching** — calculating Tanimoto, Dice, Cosine similarities, and building a high-speed chemical search engine!